# Moon Image Reduction — 2026-03-02

## Data overview

| Frame | File | EXPTIME | GAIN | OFFSET | Software |
|-------|------|---------|------|--------|----------|
| Science | `moon.fit` | 0.002 s | 41 | 50 | EZCAP_Qt |
| Dithered twilight flats | `flat0002-2s.fit` … `flat0013-2s.fit` (12) | 0.002 s | 91 | 103 | EZCAP_Qt |

No calibration frames exist at matching GAIN/OFFSET settings.

## Flat-field technique — Median dithered twilight flat

The 12 flat frames are **dithered twilight flats**. The procedure:

1. Point at the bright twilight sky
2. Expose briefly (2 ms)
3. Let the field drift — stars move to different pixels between frames
4. Repeat; median-combine all frames

The pixel-wise median rejects point sources (stars differ between frames)
and retains the spatially-smooth illumination response: vignetting, dust
shadows, and PRNU. This is sometimes called a *superflat* or *unguided
twilight flat*.

## GAIN mismatch

Flats were acquired at GAIN=91; the science frame at GAIN=41. A **normalised**
flat field corrects for pixel-to-pixel quantum efficiency (QE) and vignetting —
properties of the photodiodes and optics that are independent of the downstream
amplifier gain. Normalising each flat by its own median before combination removes
both the absolute bias level and the changing twilight brightness, leaving only the
relative sensitivity pattern.

## Calibration pipeline

| Step | Treatment |
|------|-----------|
| Bias | Estimated from dark sky corners of `moon.fit` (no matching bias frames available) |
| Dark | Negligible at 2 ms exposure |
| Flat | Median of 12 per-frame-normalised dithered twilight exposures |

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from astropy.io import fits
from astropy.visualization import AsinhStretch, ImageNormalize, PercentileInterval

%matplotlib inline

DIR = 'FITSDATA/20260302/'
os.makedirs('tmpimg',           exist_ok=True)
os.makedirs('final-imgs/FITS',  exist_ok=True)

## 1. Load science frame

In [ ]:
moon_raw = fits.getdata(DIR + 'moon.fit', memmap=False).astype(np.float32)
moon_hdr = fits.getheader(DIR + 'moon.fit')
ny, nx = moon_raw.shape

print(f'Shape : {nx} × {ny} px')
print(f'GAIN={moon_hdr["GAIN"]}, OFFSET={moon_hdr["OFFSET"]}, '
      f'EXPTIME={moon_hdr["EXPTIME"]} s')
print(f'DATE-OBS : {moon_hdr["DATE-OBS"]}')
print(f'Detector Temp: {moon_hdr["CCD-TEMP"]} °C')
print(f'ADU range: {moon_raw.min():.0f} – {moon_raw.max():.0f}')

fig, ax = plt.subplots(figsize=(11, 7))
norm = ImageNormalize(moon_raw, interval=PercentileInterval(99.5),
                      stretch=AsinhStretch(a=0.02))
im = ax.imshow(moon_raw, origin='upper', cmap='gray', norm=norm)
ax.set_title('Raw Moon Frame — 2 ms, GAIN 41')
ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='4%', pad=0.05)
fig.colorbar(im, cax=cax, label='ADU')
plt.tight_layout()
plt.savefig('tmpimg/moon_raw.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Bias estimation from dark sky corners

At 2 ms, dark current is negligible. The four 200 × 200 px corners lie
outside the lunar disk and carry essentially zero sky signal, so their
median is a direct measurement of the electronic offset at GAIN=41.

In [ ]:
patch = 200
corners = {
    'TL': moon_raw[:patch, :patch],
    'TR': moon_raw[:patch, -patch:],
    'BL': moon_raw[-patch:, :patch],
    'BR': moon_raw[-patch:, -patch:],
}

print('Corner statistics (200 × 200 px):')
corner_medians = []
for label, patch_data in corners.items():
    med = float(np.median(patch_data))
    std = float(np.std(patch_data))
    corner_medians.append(med)
    print(f'  {label}: median = {med:.1f} ADU,  std = {std:.2f} ADU')

# Robust aggregate: median of corner medians
bias_level = float(np.median(corner_medians))
print(f'\nBias level (median of corners): {bias_level:.1f} ADU')

## 3. Build master flat from dithered twilight frames

Each frame is normalised by its own median before stacking so that the
changing twilight brightness does not bias the combined flat. Stars that
drifted between exposures are eliminated by the pixel-wise median.

In [ ]:
flat_files = sorted(glob.glob(DIR + 'flat0*-2s.fit'))
n_flats = len(flat_files)
print(f'Dithered twilight flats: {n_flats} frames')
print(f'  GAIN=91, OFFSET=103, EXPTIME=0.002 s\n')

flat_cube = np.empty((n_flats, ny, nx), dtype=np.float32)
for i, f in enumerate(flat_files):
    raw = fits.getdata(f, memmap=False).astype(np.float32)
    med = np.median(raw)
    flat_cube[i] = raw / med
    hdr = fits.getheader(f)
    print(f'  {os.path.basename(f)}: median = {med:.0f} ADU  '
          f'({hdr["DATE-OBS"]}, Det T={hdr["CCD-TEMP"]} °C)')

master_flat = np.median(flat_cube, axis=0)
del flat_cube

print(f'\nMaster flat: mean = {master_flat.mean():.4f},'
      f'  std = {master_flat.std():.4f}')
print(f'  range: {master_flat.min():.3f} – {master_flat.max():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Flat image
im0 = axes[0].imshow(master_flat, origin='upper', cmap='inferno',
                     vmin=np.percentile(master_flat, 1),
                     vmax=np.percentile(master_flat, 99))
axes[0].set_title('Master flat (median of 12 dithered twilight frames)')
axes[0].set_xlabel('x (px)'); axes[0].set_ylabel('y (px)')
div0 = make_axes_locatable(axes[0])
fig.colorbar(im0, cax=div0.append_axes('right', size='4%', pad=0.05),
             label='Normalised response')

# Cross-section profiles through centre
cy, cx = ny // 2, nx // 2
axes[1].plot(master_flat[cy, :], lw=0.6, color='steelblue', label='Row (y = centre)')
axes[1].plot(master_flat[:, cx], lw=0.6, color='firebrick', alpha=0.8,
             label='Column (x = centre)')
axes[1].axhline(1.0, ls='--', color='gray', lw=0.8)
axes[1].set_xlabel('Pixel'); axes[1].set_ylabel('Normalised response')
axes[1].set_title('Flat field cross-sections through centre')
axes[1].legend()

plt.tight_layout()
plt.savefig('tmpimg/moon_master_flat.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Star rejection verification

Compare one raw normalised flat against the master to confirm that stars
that appeared in individual frames are absent from the median-combined result.

In [ ]:
single_raw = fits.getdata(flat_files[0], memmap=False).astype(np.float32)
single_norm = single_raw / np.median(single_raw)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

vmin, vmax = np.percentile(single_norm, [1, 99])

axes[0].imshow(single_norm, origin='upper', cmap='gray', vmin=vmin, vmax=vmax)
axes[0].set_title(f'Single flat (normalised)\n{os.path.basename(flat_files[0])}')

axes[1].imshow(master_flat, origin='upper', cmap='gray', vmin=vmin, vmax=vmax)
axes[1].set_title('Master flat (median of 12) — stars rejected')

diff = single_norm - master_flat
vd = float(np.percentile(np.abs(diff), 99.5))
im2 = axes[2].imshow(diff, origin='upper', cmap='RdBu_r', vmin=-vd, vmax=vd)
axes[2].set_title('Difference (single − master)\nRejected point sources visible')
div2 = make_axes_locatable(axes[2])
fig.colorbar(im2, cax=div2.append_axes('right', size='4%', pad=0.05))

for ax in axes:
    ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')

plt.tight_layout()
plt.savefig('tmpimg/moon_star_rejection.png', dpi=150, bbox_inches='tight')
plt.show()
del single_raw, single_norm, diff

## 5. Reduce science frame

$$\text{moon}_{\text{reduced}} = \frac{\text{moon} - \text{bias}}{\text{flat}}$$

Pixels where the master flat is near zero (bad / dust-blocked) are
replaced with unity in the denominator to avoid divide-by-zero artefacts.

In [ ]:
flat_safe = np.where(master_flat > 0.5, master_flat, 1.0)
moon_reduced = np.clip((moon_raw - bias_level) / flat_safe, 0.0, None)

sat_mask = moon_raw >= 65534
n_sat = int(np.sum(sat_mask))

print(f'Reduced moon ADU range: {moon_reduced.min():.0f} – {moon_reduced.max():.0f}')
print(f'Saturated pixels: {n_sat:,} ({n_sat / moon_raw.size * 100:.2f} %)')

## 6. Results

In [ ]:
norm_shared = ImageNormalize(moon_reduced,
                             interval=PercentileInterval(99.9),
                             stretch=AsinhStretch(a=0.01))

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(moon_raw - bias_level, origin='upper', cmap='gray', norm=norm_shared)
axes[0].set_title('Bias-subtracted only')
axes[0].set_xlabel('x (px)'); axes[0].set_ylabel('y (px)')

axes[1].imshow(moon_reduced, origin='upper', cmap='gray', norm=norm_shared)
axes[1].set_title('Bias-subtracted + flat-fielded')
axes[1].set_xlabel('x (px)'); axes[1].set_ylabel('y (px)')

plt.suptitle('Moon — 2026-03-02 20:27 UT  |  QHY268M  |  2 ms  |  GAIN 41',
             fontsize=13)
plt.tight_layout()
plt.savefig('final-imgs/moon_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
vmax_disp = float(np.percentile(moon_reduced[~sat_mask], 99.9))

fig, ax = plt.subplots(figsize=(13, 9))
im = ax.imshow(moon_reduced, origin='upper', cmap='gray',
               norm=PowerNorm(gamma=0.4, vmin=0, vmax=vmax_disp))
ax.set_title(
    'Moon — 2026-03-02 20:27 UT\n'
    'QHY268M (IMX571)  |  2 ms  |  GAIN 41  |  dithered twilight flat applied',
    fontsize=12)
ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')
divider = make_axes_locatable(ax)
cax = divider.append_axes('right', size='3%', pad=0.05)
fig.colorbar(im, cax=cax, label='ADU (bias-subtracted)')
plt.tight_layout()
plt.savefig('final-imgs/moon_reduced.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Flat-field correction — detail view

Zoom into a limb region where vignetting is most pronounced to show
the correction applied.

In [ ]:
# Centre-of-disk crop for crater detail
threshold = bias_level + 50
ys, xs = np.where(moon_raw > threshold)
moon_cy = int(np.median(ys)) if len(ys) else ny // 2
moon_cx = int(np.median(xs)) if len(xs) else nx // 2

half = 700   # 1400 x 1400 px crop centred on disk
zy = slice(max(0, moon_cy - half), min(ny, moon_cy + half))
zx = slice(max(0, moon_cx - half), min(nx, moon_cx + half))

raw_zoom  = moon_raw[zy, zx] - bias_level
red_zoom  = moon_reduced[zy, zx]
flat_zoom = master_flat[zy, zx]

vmax_z = float(np.percentile(red_zoom, 99.5))
norm_z = PowerNorm(gamma=0.4, vmin=0, vmax=vmax_z)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

axes[0].imshow(raw_zoom,  origin='upper', cmap='gray', norm=norm_z)
axes[0].set_title('Bias-subtracted only — centre crop')

axes[1].imshow(red_zoom,  origin='upper', cmap='gray', norm=norm_z)
axes[1].set_title('Flat-fielded — centre crop')

im2 = axes[2].imshow(flat_zoom, origin='upper', cmap='RdBu_r',
                     vmin=np.percentile(flat_zoom, 1),
                     vmax=np.percentile(flat_zoom, 99))
axes[2].set_title('Flat field — this region')
div2 = make_axes_locatable(axes[2])
fig.colorbar(im2, cax=div2.append_axes('right', size='4%', pad=0.05),
             label='Normalised response')

for ax in axes:
    ax.set_xlabel('x (px)'); ax.set_ylabel('y (px)')

plt.suptitle('Crater detail — centre of lunar disk', fontsize=13)
plt.tight_layout()
plt.savefig('final-imgs/moon_zoom_flatfield.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save calibrated frame as FITS
out_hdr = moon_hdr.copy()

# float32 (BITPIX=-32) needs no BZERO/BSCALE offset coding
for kw in ('BZERO', 'BSCALE'):
    if kw in out_hdr:
        del out_hdr[kw]

out_hdr['HISTORY'] = 'Bias subtracted: level estimated from 4x 200x200 px sky corners'
out_hdr['HISTORY'] = f'Bias level: {bias_level:.1f} ADU (GAIN={moon_hdr["GAIN"]}, OFFSET={moon_hdr["OFFSET"]})'
out_hdr['HISTORY'] = f'Flat field: median of {n_flats} dithered twilight flats (flat0002-flat0013-2s.fit)'
out_hdr['HISTORY'] = 'Flat frames: GAIN=91, OFFSET=103, 0.002 s, normalised per frame before median'
out_hdr['HISTORY'] = 'PRNU/vignetting correction valid; gain-dependent non-linearity uncharacterised'
out_hdr['HISTORY'] = 'Dark current not subtracted (negligible at 0.002 s)'
out_hdr['BIASLVL']  = (bias_level, '[ADU] estimated bias level')
out_hdr['NFLATS']   = (n_flats,    'number of flat frames combined')
out_hdr['FLATGAIN'] = (91,         'GAIN setting of flat frames')

fits_path = 'final-imgs/FITS/moon_reduced.fit'
fits.writeto(fits_path, moon_reduced, out_hdr, overwrite=True)

print(f'Saved: {fits_path}')
print(f'  BITPIX  = -32  (float32)')
print(f'  NAXIS1  = {nx},  NAXIS2 = {ny}')
print(f'  ADU range: {moon_reduced.min():.1f} -- {moon_reduced.max():.1f}')

## 8. Reduction summary

In [ ]:
print('=' * 65)
print('  MOON IMAGE REDUCTION — 2026-03-02')
print('=' * 65)
print(f'  Science frame  : moon.fit')
print(f'  DATE-OBS       : {moon_hdr["DATE-OBS"]}')
print(f'  Exposure       : {moon_hdr["EXPTIME"]} s')
print(f'  GAIN / OFFSET  : {moon_hdr["GAIN"]} / {moon_hdr["OFFSET"]}')
print(f'  Detector Temp  : {moon_hdr["CCD-TEMP"]} °C')
print()
print(f'  Flat field     : median dithered twilight flat ({n_flats} frames)')
print(f'  Flat GAIN      : 91  (normalised flat; PRNU/vignetting are gain-independent)')
print(f'  Flat σ         : {master_flat.std():.4f}  (PRNU + vignetting combined)')
print(f'  Flat range     : {master_flat.min():.3f} – {master_flat.max():.3f}')
print()
print(f'  Bias level     : {bias_level:.1f} ADU  (estimated from dark sky corners)')
print(f'  Dark current   : not subtracted (negligible at 2 ms)')
print()
print(f'  Reduced signal : {moon_reduced.min():.0f} – {moon_reduced.max():.0f} ADU')
print(f'  Saturated px   : {n_sat:,}  ({n_sat / moon_raw.size * 100:.2f} %)')
print()
print('  Limitations:')
print('    Flat taken at GAIN 91 vs science GAIN 41; normalised correction')
print('    is valid for PRNU/vignetting but gain-dependent non-linearity')
print('    residuals at extreme ADU values are uncharacterised.')
print('    Brightest lunar highlands saturated (65 535 ADU).')
print('    Bias estimated from corners, not from dedicated bias frames.')
print('=' * 65)